# 00 — Prepare the study data

This is the first of three notebooks. Here you build the dataset that notebooks 01 and 02 analyse.

## The causal question

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

## The words we will use

Causal inference has its own vocabulary. Four terms cover most of this notebook:

| Term | Plain meaning | Here |
|---|---|---|
| **Unit** | One thing you observe, and could apply the treatment to. | One code example (one row of the table). |
| **Treatment** | The action whose effect you want to measure — the *cause* side of the question. | Applying the backdoor defense. |
| **Outcome** | The result you measure afterwards — the *effect* side. | Whether the backdoor was detected. |
| **Covariate** | Any other variable recorded about a unit. Some matter causally, some do not. | Code complexity, reviewer experience, and so on. |

The word "treatment" comes from medicine, where it meant a drug. It does not have to be medical. It simply means **the intervention being studied** — here, a software defense.

In this dataset, treatment and outcome are both 0 or 1:

| Variable | Meaning |
|---|---|
| `treatment = 0` | baseline random filtering |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

Because `outcome` is only ever 0 or 1, its average is just the fraction of successes — the **detection success rate (DSR)**. A DSR of 0.70 means 70% of examples were detected.

**Tutorial path:** **00 Data preparation** → 01 Correlational analysis → 02 Causal inference


## 1. What is real and what is synthetic

The raw CSV holds real code and docstring examples. They give the dataset realistic variation from one unit to the next.

The treatment assignment and the detection outcome are **synthetic** — produced by a program rather than measured in a real study. Treat the numbers as a teaching exercise, not as evidence about any actual backdoor defense.

Each row of the final table is one code example, observed under **one** treatment condition, with **one** detection outcome.


## 2. Configure the preparation

Set the input file, the output path, and the random seed used by this notebook.

The seed fixes the random number generator so that everyone running this notebook gets the same dataset.


In [ ]:
from src.causal_data_prep import (
    DEFAULT_COVARIATES,
    engineer_features,
    load_source_data,
    make_synthetic_observational_data,
    save_causal_dataset,
    validate_causal_dataset,
)

def default_params():
    return {
        "source_dataset": "data/raw_code.csv",
        "lizard_cache_folder": "cache/lizard",
        "causal_dataset": "data/causal_data.csv",
        "random_seed": 42,
        "covariate_columns": DEFAULT_COVARIATES,
    }

params = default_params()
params


## 3. Load the source examples

The source file contains:

| Column | Meaning |
|---|---|
| `input_code` | the code example |
| `output_docstring` | its docstring |
| `reviewer_experience` | how experienced the assigned reviewer is |
| `rollout_eligibility` | whether the example is eligible for the defense rollout |
| `noise_feature` | a background variable with no intended meaning |

The last three are **pre-treatment variables**: their values are already fixed *before* anyone decides whether to apply the defense.

Timing will matter a great deal later. A variable that existed before the treatment decision could have influenced that decision. A variable created afterwards could not — nothing can cause something that already happened.


In [ ]:
source_df = load_source_data(params["source_dataset"])

print(f"Loaded {len(source_df):,} source examples.")
source_df[
    [
        "input_code",
        "output_docstring",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].head()


## 4. Engineer code features

Derive four readable characteristics of each code example:

| Feature | What it counts |
|---|---|
| `code_number_tokens` | size of the code, in tokens |
| `code_complexity` | how branch-heavy the control flow is |
| `code_num_identifiers` | how many distinct names are used |
| `code_num_strings` | how many string literals appear |

These are **measurements**: numbers that describe a unit. A measurement carries no causal role on its own. Whether `code_complexity` causes anything is a question you answer by reasoning in notebook 02 — it is not settled by the column existing in the data.


In [ ]:
feature_df = engineer_features(
    source_df,
    cache_dir=params["lizard_cache_folder"],
)

feature_df[
    [
        "code_number_tokens",
        "code_complexity",
        "code_num_identifiers",
        "code_num_strings",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].describe().T


## 5. Generate the observed treatment and outcome

Each unit is assigned to exactly one condition:

- `0` — random filtering, the baseline, also called the **control** condition;
- `1` — backdoor defense applied, the **treated** condition.

`outcome` then records whether detection succeeded for that unit.

### The fundamental problem

Each unit receives one condition and produces one result. We never see what *would have happened* to that same unit under the other condition. That unobserved alternative is called a **counterfactual**, and its absence is the central difficulty of causal inference. If we could see both, we could simply subtract and be done.

### Two variables measured after treatment

`inspection_intensity` and `manual_review_flag` are recorded **after** the treatment decision. They are included deliberately: both correlate strongly with treatment and with outcome, which makes them tempting to adjust for. Notebook 02 shows why giving in to that temptation can make the answer worse rather than better.


In [ ]:
causal_df, study_info = make_synthetic_observational_data(
    feature_df,
    seed=params["random_seed"],
)

print(f"Backdoor-defense prevalence: {study_info.treatment_prevalence:.3f}")
print(f"Observed detection success rate: {study_info.outcome_prevalence:.3f}")

causal_df.head()


## 6. Validate the table

Check that:

- every unit appears exactly once;
- both treatment conditions are present — with no control group there is nothing to compare against;
- `outcome` is binary, meaning it only ever takes the value 0 or 1;
- the analysis variables are numeric and finite;
- no analysis values are missing.


In [ ]:
validation_summary = validate_causal_dataset(
    causal_df,
    covariates=params["covariate_columns"],
)

validation_summary


## 7. Save the dataset

This cell writes `data/causal_data.csv`, the one file that notebooks 01 and 02 read.

It holds one row per code example: the observed treatment, the observed outcome, and every covariate.


In [ ]:
data_path = save_causal_dataset(
    causal_df,
    params["causal_dataset"],
)

print(f"Saved causal dataset: {data_path}")


## Next: notebook 01

Notebook 00 answered: **what was observed for each unit?**

Notebook 01 asks something different: **what relationships can we see in the observed data?**

That is a question about correlation, not yet about cause. Keeping those two apart is the point of this tutorial.
